In [ ]:
import sys
sys.path.append('C:/Users/sabri/repositorios/bitbirch')

import pandas as pd
import numpy as np
from rdkit import Chem
from rdkit.Chem import AllChem
from bitbirch import BitBirch

# Function to calculate ECFP4 fingerprint
def calc_ecfp4(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol:
        # Usando GetMorganFingerprintAsBitVect para ECFP4
        return AllChem.GetMorganFingerprintAsBitVect(mol, 2, nBits=1024) 
    else:
        return None

# Load datasets
file1 = 'data/3T3_curated_reduced_1-5.csv'  
file2 = 'data/HEK_curated_reduced_1-5.csv'  

df1 = pd.read_csv(file1)
df2 = pd.read_csv(file2)

# Add columns 'cell_type' and 'balanced_type'
df1['cell_type'] = '3T3'
df1['balanced_type'] = '1-5'  
df2['cell_type'] = 'HEK293'
df2['balanced_type'] = '1-5'  

# Concatenate dataframes
df = pd.concat([df1, df2], ignore_index=True)

# Calculate ECFP4 descriptor for each SMILES
df['ecfp4'] = df['SMILES'].apply(calc_ecfp4)

# Remove lines were the descriptor could not be calculated
df = df.dropna(subset=['ecfp4'])

# Convert the RDKit fingerprint to a numpy array
ecfp4_vectors = np.array(df['ecfp4'].tolist())  

# Clustering using BitBirch
bitbirch = BitBirch()
bitbirch.fit(ecfp4_vectors)

# Obtain the cluster IDs for each molecule
cluster_mol_ids = bitbirch.get_cluster_mol_ids()

# Create a list to store the cluster assignment for each molecule
cluster_assignment = [-1] * len(df)  

# Map the cluster IDs to the molecule indices
for cluster_id, mol_indices in enumerate(cluster_mol_ids):
    for mol_index in mol_indices:
        cluster_assignment[mol_index] = cluster_id

# Add the cluster ID to the dataframe
df['cluster_id'] = cluster_assignment

# Mantain only the relevant columns
df = df[['SMILES', 'ecfp4', 'Outcome', 'PUBCHEM_CID', 'cell_type', 'balanced_type', 'cluster_id']]

# Save the clustered data
df.to_csv('data/clustered_data.csv', index=False)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Function to calculate the similarity of outcomes in a cluster
def calc_outcome_similarity(data):
    unique_outcomes = set(data['Outcome'])
    if {0.0, 1.0}.issubset(unique_outcomes):  
        return 'different'
    return 'same'

# Aply the function to the data
cluster_outcomes = (
    df.groupby(['cell_type', 'balanced_type', 'cluster_id'], group_keys=False)
      .apply(lambda group: pd.Series({'outcome_similarity': calc_outcome_similarity(group)}))
      .reset_index()
)

# Count the number of clusters with the same and different outcomes
outcome_stats = (
    cluster_outcomes.groupby(['cell_type', 'balanced_type', 'outcome_similarity'])['cluster_id']
    .nunique()
    .reset_index()
)

outcome_stats = (
    outcome_stats.pivot_table(
        index=['cell_type', 'balanced_type'],
        columns='outcome_similarity',
        values='cluster_id',
        fill_value=0
    )
    .reset_index()
)

outcome_stats.rename(columns={'different': 'clusters_with_different_outcomes', 'same': 'clusters_with_same_outcomes'}, inplace=True)

# Calculate the total number of clusters
total_clusters = df.groupby(['cell_type', 'balanced_type'])['cluster_id'].nunique().reset_index()
total_clusters.rename(columns={'cluster_id': 'total_clusters'}, inplace=True)

# Merge the dataframes
summary = pd.merge(total_clusters, outcome_stats, on=['cell_type', 'balanced_type'])

# Print the summary
print("Total number of clusters, with same and different outcomes:")
for _, row in summary.iterrows():
    print(f"Cell Type: {row['cell_type']} - Balanced Type: {row['balanced_type']}")
    print(f"  Total Clusters: {row['total_clusters']}")
    print(f"  Clusters with Same Outcomes: {row['clusters_with_same_outcomes']}")
    print(f"  Clusters with Different Outcomes: {row['clusters_with_different_outcomes']}")
    print("")

# Prepare the data for plotting
plot_data = pd.melt(summary, 
                    id_vars=['cell_type', 'balanced_type'], 
                    value_vars=['clusters_with_same_outcomes', 'clusters_with_different_outcomes'], 
                    var_name='outcome_type', 
                    value_name='count')

# Rename the outcome types labels
plot_data['outcome_type'] = plot_data['outcome_type'].map({
    'clusters_with_same_outcomes': 'Homogeneous Outcomes',
    'clusters_with_different_outcomes': 'Heterogeneous Outcomes'
})

# Create the bar plot
plt.figure(figsize=(12, 7))
sns.barplot(
    data=plot_data,
    x='cell_type',
    y='count',
    hue='outcome_type',
    dodge=True,
    palette='Set2'
)

# Customize the plot
plt.title('Cluster Outcome Analysis by Cell Type', fontsize=16)
plt.ylabel('Number of Clusters', fontsize=14)
plt.xlabel('Cell Type', fontsize=14) 
plt.xticks(
    ticks=range(len(summary['cell_type'].unique())), 
    labels=summary['cell_type'].unique(),
    rotation=45,
    ha='right',
    fontsize=12
)
plt.legend(title='Outcome Type', title_fontsize=12, fontsize=10)
plt.tight_layout()

# Save the plot
plt.savefig('cluster_outcome_analysis.png', format='png', dpi=300)

# Plot the bar chart
plt.show()